# 训练 LLaVA

在 [准备CC3M数据集和数据加载器.ipynb](准备CC3M数据集和数据加载器.ipynb) 中，我们为我们的 LLaVA 模型创建了 Dataset 和 Collator，已经能够便利地获得训练需要用到的数据批量。本节，我们讲按照 [准备CC3M数据集和数据加载器.ipynb](准备CC3M数据集和数据加载器.ipynb) 描述的方案进行模型训练。

训练方式：

- 视觉塔（CLIP ViT-L/14-336）冻结；
- LLM（Qwen1.5-4B-Chat）用 LoRA 训练；
- 视觉-文本投影层（`multi_modal_projector`，两层 MLP）全量训练。

In [ ]:
import os

# 模型和数据都在本地，关闭 Hub 访问：
# 避免 PEFT 在保存 checkpoint 时把相对路径 show_model/model001 当成 Hub repo id 去联网重试
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

# DataLoader fork 前禁用 tokenizers 并行，消掉 fork 警告
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# 避免 libgomp 对 OMP_NUM_THREADS 非法值的警告；可按 CPU 核数调整
os.environ["OMP_NUM_THREADS"] = "8"

## 加载 Dataset 和 Collator

In [ ]:
# 从独立的 llava_data 模块加载数据相关常量、Dataset 和 Collator
from transformers import LlavaProcessor

from llava_data import (
    DATA_ROOT,
    IMAGE_DIR,
    MODEL_DIR,
    LlavaPretrainDataset,
    TrainCollator,
)

In [ ]:
processor = LlavaProcessor.from_pretrained(MODEL_DIR)

# 使用 Dataset 加载训练数据集和测试数据集
train_dataset = LlavaPretrainDataset(DATA_ROOT / "chat_train.json", IMAGE_DIR, processor)
val_dataset = LlavaPretrainDataset(DATA_ROOT / "chat_val.json", IMAGE_DIR, processor)

## 加载模型并接入 LoRA

CLIP 原论文在预训练和下游迁移中都冻结视觉塔，所以这里要保证 **CLIP 视觉塔全程冻结**。PEFT 的 `target_modules=["q_proj", ...]` 是按模块名后缀匹配的，CLIP 的 attention 里也有 `q_proj/k_proj/v_proj`，不加限制就会把 LoRA 误注入视觉塔。

In [ ]:
from peft import LoraConfig, get_peft_model
from transformers import LlavaForConditionalGeneration